In [ ]:
import xml.etree.ElementTree as ET

# Charger le fichier XML
tree = ET.parse('/home/kory/Projet_DST_exploration/exploration/data/00043445_00043449_ocr.xml')
root = tree.getroot()

# Namespace pour accéder correctement aux éléments du fichier XML
namespace = {'ns': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

# Extraire le nom du fichier image
image_filename = root.find('.//ns:Page', namespace).attrib['imageFilename']

# Extraire les coordonnées des régions de texte
regions = []
for region in root.findall('.//ns:TextRegion', namespace):
    coords = region.find('.//ns:Coords', namespace).attrib['points']
    regions.append(coords)

print(f"Nom du fichier image: {image_filename}")
print("Coordonnées des régions de texte:", regions)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Charger l'image
image_path = '/home/kory/Projet_DST_exploration/exploration/data/00043445_00043449.tif'
image = cv2.imread(image_path)

# Convertir l'image en RGB (OpenCV charge en BGR par défaut)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Dessiner les régions de texte sur l'image
for region in regions:
    points = [tuple(map(int, point.split(','))) for point in region.split()]
    points_array = np.array([points], np.int32)  # OpenCV attend un numpy array de int32
    cv2.polylines(image_rgb, [points_array], isClosed=True, color=(255, 0, 0), thickness=2)  # Utiliser couleur rouge

# Afficher l'image avec les annotations
plt.figure(figsize=(10, 15))  # Ajuster la taille si nécessaire
plt.imshow(image_rgb)
plt.axis('off')  # Désactiver les axes
plt.show()


In [ ]:
import xml.etree.ElementTree as ET

# Charger le fichier ground truth XML
tree_gt = ET.parse('/home/kory/Projet_DST_exploration/exploration/data/00043445_00043449_gt.xml')
root_gt = tree_gt.getroot()

# Namespace pour accéder correctement aux éléments du fichier XML
namespace = {'ns': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

# Extraire les régions de texte et leurs entités
gt_regions = []
for region in root_gt.findall('.//ns:TextRegion', namespace):
    coords = region.find('.//ns:Coords', namespace).attrib['points']
    entity = region.find('.//ns:Property', namespace).attrib.get('value', 'unknown')
    gt_regions.append((coords, entity))

# Afficher les coordonnées et les entités
for coords, entity in gt_regions:
    print(f"Entity: {entity}, Coords: {coords}")


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Charger l'image
image_path = '/home/kory/Projet_DST_exploration/exploration/data/00043445_00043449.tif'
image = cv2.imread(image_path)

# Convertir l'image en RGB
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Définir des couleurs pour chaque entité
entity_colors = {
    'supplier': (255, 0, 0),    # Rouge
    'receiver': (0, 255, 0),    # Vert
    'invoice_info': (0, 0, 255),  # Bleu
    'positions': (255, 255, 0),  # Jaune
    'other': (255, 0, 255)      # Magenta
}

# Dessiner les régions de texte sur l'image en fonction de l'entité
for coords, entity in gt_regions:
    points = [tuple(map(float, point.split(','))) for point in coords.split()]
    points_array = np.array([points], np.int32)
    color = entity_colors.get(entity, (255, 255, 255))  # Blanc par défaut si l'entité n'est pas reconnue
    cv2.polylines(image_rgb, [points_array], isClosed=True, color=color, thickness=2)

# Afficher l'image avec les annotations
plt.figure(figsize=(10, 15))
plt.imshow(image_rgb)
plt.axis('off')  # Désactiver les axes
plt.show()
